# Single chain dataset

In [8]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from types import SimpleNamespace

from cli import Cli
from dataset_generator.types import DATASET_TYPE

# Define the absolute path for the root directory for outputs here (and if possible, make sure to have a prepared _dataset folder)
OUTPUTS_ROOT = Path("/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/deepsad-cctx-eval-ablationsemfusion-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge")

In [9]:
# Obtain all relevant run directories that match the pattern and contain a params.yaml file
run_dirs = sorted(
    p for p in OUTPUTS_ROOT.glob("train_epochs100_augment0_lr0.002*")
    if p.is_dir() and (p / "params.yaml").exists()
)

print(f"Found {len(run_dirs)} run directories")
for p in run_dirs:
    print(p)

Found 2 run directories
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/deepsad-cctx-eval-ablationsemfusion-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_es15_deepsad_ablationsemfusion__202608171445
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/deepsad-cctx-eval-ablationsemfusion-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_nosf_es15_deepsad_ablationsemfusion__202608171107


In [ ]:
import yaml
import pandas as pd

def load_run_label(run_dir: Path) -> str:
    with open(run_dir / "params.yaml") as f:
        doc = yaml.safe_load(f)
    rm_semantic_fusion = doc["params"].get("rm_semantic_fusion", False)
    return "Without Semantic Fusion" if rm_semantic_fusion else "With Semantic Fusion"

def load_avg_losses(run_dir: Path) -> pd.DataFrame:
    # Only fold_1 is used here (no cross-fold aggregation) to inspect a single fold's curve directly.
    train_df = pd.read_csv(run_dir / "val_train_loss.csv").set_index("epoch")
    val_df = pd.read_csv(run_dir / "val_val_loss.csv").set_index("epoch")
    return pd.DataFrame({
        "train_loss": train_df["fold_1"].dropna(),
        "val_loss": val_df["fold_1"].dropna(),
    })

runs = [
    {"run_dir": p, "label": load_run_label(p), "losses": load_avg_losses(p)}
    for p in run_dirs
]
for r in runs:
    print(f"{r['label']}: {len(r['losses'])} epochs")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(runs), figsize=(6 * len(runs), 4), sharey=True)
if len(runs) == 1:
    axes = [axes]

for ax, r in zip(axes, runs):
    losses = r["losses"]
    ax.plot(losses.index, losses["train_loss"], label="Training Loss", color="tab:blue")
    ax.plot(losses.index, losses["val_loss"], label="Validation Loss", color="tab:orange")
    ax.set_title(r["label"])
    ax.set_xlabel("Epoch")
    ax.legend()

axes[0].set_ylabel("Loss (fold 1)")
fig.suptitle("Training vs. Validation Loss")
fig.tight_layout()
plt.show()